# RNN Case Study: Time Series Forecasting

## Project Overview

In this case study, you will apply everything you've learned about RNNs to build a complete time series forecasting pipeline. We'll use **Air Quality data** to predict pollution levels.

### Learning Objectives
- Build an end-to-end RNN time series forecasting pipeline
- Handle multivariate time series data
- Compare SimpleRNN, LSTM, and GRU architectures
- Apply proper train/validation/test splitting for time series
- Evaluate forecasting models with appropriate metrics
- Handle real-world challenges (missing data, seasonality)

### Dataset: Air Quality
- **Features**: Hourly sensor readings for various pollutants and weather conditions
- **Target**: CO(GT) - Carbon Monoxide concentration
- **Time Period**: 1 year of hourly data

### Prerequisites
- Completed notebooks 01-04 in this RNN module
- Understanding of RNNs, LSTMs, and GRUs

---

# RNN case study: air-quality forecasting — Assessment (D025)

**Time:** 60 minutes · **Marks:** 100 · **Format:** coding notebook plus written justification, marked by a human.

**Data:** `generated inside the notebook (synthetic air-quality series), no download`

**Rules:** only what has been taught up to this day; comment the *why*, not the *what*; a printed number without its justification earns half the marks.

Tasks 8, 10, 12, 15, Part 7 and the four exercises are bonus. The instructor copy trains for 5 epochs.

| Task | Bloom level | Marks | What is marked |
|---|---|---|---|
| Tasks 1 to 4: explore the series | understand | 10 | Descriptives, the series plotted, the daily cycle shown by hour, the correlation matrix read for the strongest pair. |
| Tasks 5 to 7: scale, window, split in time | apply | 20 | Scaler fitted on the training period only; sequences built with a stated window; the split chronological, never random. |
| Task 9: LSTM model | apply | 15 | An LSTM layer with the input shape (timesteps, features), a Dense output, MSE loss. |
| Task 11: train with callbacks | apply | 10 | EarlyStopping on validation loss; the curves plotted. |
| Tasks 13 to 14: evaluate and compare | analyse | 15 | Test RMSE and MAE in the original units (inverse-scaled); the models compared on a bar chart with the naive 'last value' baseline beside them. |
| Task 16: error analysis | analyse | 10 | Residuals over time; when the model is worst (peaks, regime changes) named. |
| Task 18: conclusions | evaluate | 20 | Whether the RNN beat the naive baseline and by how much; what the window length assumed; what a deployment would need (retraining cadence, drift). |

Anything headed **Bonus (not marked)** is optional stretch material.

## Part 1: Data Loading and Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Create synthetic air quality dataset
def create_air_quality_data(n_samples=8760):  # 1 year of hourly data
    """Create synthetic air quality dataset."""
    np.random.seed(42)
    
    # Time index
    dates = pd.date_range(start='2004-03-10', periods=n_samples, freq='h')
    hour = dates.hour
    day_of_week = dates.dayofweek
    
    # Temperature: Daily cycle + seasonal trend + noise
    temp_daily = 10 * np.sin(2 * np.pi * hour / 24)
    temp_seasonal = 10 * np.sin(2 * np.pi * np.arange(n_samples) / (24*365))
    temperature = 15 + temp_daily + temp_seasonal + np.random.normal(0, 2, n_samples)
    
    # Humidity: Inverse relationship with temperature
    humidity = 70 - 0.5 * temperature + np.random.normal(0, 5, n_samples)
    humidity = np.clip(humidity, 20, 95)
    
    # CO concentration: Traffic patterns (rush hours) + temperature effect
    rush_hour = np.where((hour >= 7) & (hour <= 9) | (hour >= 17) & (hour <= 19), 1, 0)
    weekend = np.where(day_of_week >= 5, 1, 0)
    co_base = 2.0 + 1.5 * rush_hour * (1 - 0.5 * weekend)
    co_temp_effect = -0.05 * temperature
    co = co_base + co_temp_effect + np.random.exponential(0.3, n_samples)
    co = np.clip(co, 0.5, 10)
    
    # NO2 concentration
    no2 = 0.8 * co + np.random.normal(0, 0.3, n_samples)
    no2 = np.clip(no2, 0.1, 8)
    
    df = pd.DataFrame({
        'DateTime': dates,
        'CO_GT': co,
        'NO2_GT': no2,
        'Temperature': temperature,
        'Humidity': humidity,
        'Hour': hour,
        'DayOfWeek': day_of_week
    })
    
    return df

# Create dataset
df = create_air_quality_data()
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
df.head(10)

### Task 1: Explore Basic Statistics

Display basic statistics of the dataset.

In [ ]:
# YOUR CODE HERE
# Hint: Use df.describe()



### Task 2: Visualize Time Series

Create a 3-panel plot showing:
1. CO concentration over time (first 2 weeks)
2. Temperature over time
3. Humidity over time

In [ ]:
# YOUR CODE HERE
# Hint: Use plt.subplots(3, 1, figsize=(15, 10))
# Plot data from df.iloc[:24*14] for first 2 weeks



### Task 3: Analyze Daily Patterns

Create a bar chart showing average CO concentration by hour of day.

In [ ]:
# YOUR CODE HERE
# Hint: Use df.groupby('Hour')['CO_GT'].mean()



### Task 4: Correlation Analysis

Create a correlation heatmap for the numeric features.

In [ ]:
# YOUR CODE HERE
# Hint: Use seaborn's heatmap with df[numeric_cols].corr()
import seaborn as sns



## Part 2: Data Preprocessing for Time Series

### Task 5: Scale the Features

Select features and scale them using MinMaxScaler.

In [ ]:
# Select features
features = ['CO_GT', 'NO2_GT', 'Temperature', 'Humidity', 'Hour']
target = 'CO_GT'

# YOUR CODE HERE
# Create feature matrix and scale using MinMaxScaler
data = None  # Replace with your code
scaler = None  # Replace with your code
data_scaled = None  # Replace with your code

target_idx = features.index(target)
print(f"Features: {features}")
print(f"Target: {target} (index {target_idx})")
print(f"Scaled data shape: {data_scaled.shape}")

### Task 6: Create Sequences

Implement a function to create sequences for time series forecasting.

In [ ]:
def create_sequences(data, seq_length, target_idx, forecast_horizon=1):
    """
    Create sequences for time series forecasting.
    
    Args:
        data: Scaled feature matrix
        seq_length: Number of past time steps to use
        target_idx: Index of target variable in features
        forecast_horizon: How many steps ahead to predict
    
    Returns:
        X: Input sequences (samples, seq_length, features)
        y: Target values (samples,)
    """
    # YOUR CODE HERE
    X, y = [], []
    
    # Create sequences using a loop
    # For each position i:
    #   - X should contain data[i:i+seq_length]
    #   - y should contain the target value at i+seq_length+forecast_horizon-1
    
    return np.array(X), np.array(y)

# Create sequences
SEQ_LENGTH = 24  # Use 24 hours of history
FORECAST_HORIZON = 1  # Predict 1 hour ahead

X, y = create_sequences(data_scaled, SEQ_LENGTH, target_idx, FORECAST_HORIZON)

print(f"Input shape: {X.shape}")
print(f"Target shape: {y.shape}")

### Task 7: Split Data Chronologically

**IMPORTANT**: For time series, we must split chronologically (no shuffling!).

Split into 70% train, 15% validation, 15% test.

In [ ]:
# YOUR CODE HERE
# Split data chronologically (no shuffling!)
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

X_train = None  # Replace with your code
y_train = None
X_val = None
y_val = None
X_test = None
y_test = None

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Part 3: Building RNN Models

We'll compare three architectures:
1. **SimpleRNN** - Basic recurrent architecture
2. **LSTM** - Long Short-Term Memory
3. **GRU** - Gated Recurrent Unit

### Bonus (not marked): Task 8: Build SimpleRNN Model

In [ ]:
def build_simple_rnn(input_shape):
    """Build a SimpleRNN model."""
    # YOUR CODE HERE
    # Build a model with:
    # - SimpleRNN(64, return_sequences=True)
    # - SimpleRNN(32)
    # - Dense(16, activation='relu')
    # - Dense(1)
    model = None  # Replace with your code
    return model

input_shape = (SEQ_LENGTH, len(features))
simple_rnn = build_simple_rnn(input_shape)

### Task 9: Build LSTM Model

In [ ]:
def build_lstm(input_shape):
    """Build an LSTM model."""
    # YOUR CODE HERE
    # Build a model with:
    # - LSTM(64, return_sequences=True)
    # - Dropout(0.2)
    # - LSTM(32)
    # - Dropout(0.2)
    # - Dense(16, activation='relu')
    # - Dense(1)
    model = None  # Replace with your code
    return model

lstm_model = build_lstm(input_shape)

### Bonus (not marked): Task 10: Build GRU Model

In [ ]:
def build_gru(input_shape):
    """Build a GRU model."""
    # YOUR CODE HERE
    # Build a model with similar structure to LSTM but using GRU layers
    model = None  # Replace with your code
    return model

gru_model = build_gru(input_shape)

print("Model Architectures Created:")
print(f"  SimpleRNN: {simple_rnn.count_params():,} parameters")
print(f"  LSTM: {lstm_model.count_params():,} parameters")
print(f"  GRU: {gru_model.count_params():,} parameters")

## Part 4: Training the Models

### Task 11: Define Callbacks and Train Models

In [ ]:
# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

EPOCHS = 50
BATCH_SIZE = 32

In [ ]:
# Train SimpleRNN
print("="*50)
print("Training SimpleRNN")
print("="*50)

# YOUR CODE HERE
# Compile with optimizer='adam', loss='mse', metrics=['mae']
# Train and save history

history_simple = None  # Replace with training history

In [ ]:
# Train LSTM
print("="*50)
print("Training LSTM")
print("="*50)

# YOUR CODE HERE
history_lstm = None  # Replace with training history

In [ ]:
# Train GRU
print("="*50)
print("Training GRU")
print("="*50)

# YOUR CODE HERE
history_gru = None  # Replace with training history

## Part 5: Model Evaluation and Comparison

### Bonus (not marked): Task 12: Compare Training Histories

In [ ]:
# YOUR CODE HERE
# Create 2-panel plot comparing:
# 1. Validation Loss for all three models
# 2. Validation MAE for all three models



### Task 13: Evaluate Models on Test Set

In [ ]:
def evaluate_model(model, X_test, y_test, scaler, target_idx, model_name):
    """Evaluate model and return metrics."""
    # YOUR CODE HERE
    # 1. Get predictions
    # 2. Inverse transform to original scale
    # 3. Calculate MSE, RMSE, MAE, R2
    # 4. Return dictionary with metrics
    
    return {
        'name': model_name,
        'MSE': None,
        'RMSE': None,
        'MAE': None,
        'R2': None,
        'y_pred': None,
        'y_true': None
    }

# Evaluate all models
results = {}
for name, model in [('SimpleRNN', simple_rnn), ('LSTM', lstm_model), ('GRU', gru_model)]:
    results[name] = evaluate_model(model, X_test, y_test, scaler, target_idx, name)

# Print comparison
print("\n" + "="*70)
print("TEST SET EVALUATION")
print("="*70)
print(f"{'Model':<12} {'RMSE':>10} {'MAE':>10} {'R2':>10}")
print("-"*42)
for name, res in results.items():
    print(f"{name:<12} {res['RMSE']:>10.4f} {res['MAE']:>10.4f} {res['R2']:>10.4f}")
print("="*70)

### Task 14: Create Bar Chart Comparison

In [ ]:
# YOUR CODE HERE
# Create 2-panel bar chart comparing RMSE and R2 for all models



## Part 6: Detailed Analysis of Best Model

### Bonus (not marked): Task 15: Visualize Predictions vs Actual

In [ ]:
# YOUR CODE HERE
# 1. Find the best model based on RMSE
# 2. Plot first 200 predicted vs actual values



### Task 16: Error Analysis

In [ ]:
# YOUR CODE HERE
# 1. Calculate prediction errors
# 2. Create histogram of errors
# 3. Plot errors over time
# 4. Print mean and std of errors



## Part 7: Multi-Step Forecasting (Bonus)

### Bonus (not marked): Task 17: Build and Train Multi-Step Model

In [ ]:
# YOUR CODE HERE
# Build an LSTM model that predicts the next 6 hours
# Hint: The output layer should have 6 units instead of 1



## Part 8: Conclusions

### Task 18: Write Your Conclusions

Based on your experiments, answer the following questions:

1. Which model performed best and why?
2. What are the key differences between SimpleRNN, LSTM, and GRU?
3. Why is chronological splitting important for time series?
4. What challenges did you observe with multi-step forecasting?

*Write your answers below:*

**Your Conclusions:**

1. Best model: 

2. Model differences: 

3. Chronological splitting: 

4. Multi-step challenges: 


---

## Bonus Exercises

### Bonus (not marked): Exercise 1: Different Sequence Lengths
Try sequence lengths of 12, 48, and 72 hours. How does it affect performance?

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 2: Bidirectional LSTM
Build a bidirectional LSTM model. Does it improve forecasting accuracy?

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 3: Add More Features
Include day of week as a feature. Does it help capture weekly patterns?

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 4: Attention Mechanism
Add an attention layer to the LSTM model. Does it improve performance on longer sequences?

In [ ]:
# YOUR CODE HERE

